# 🍽️ Proyecto Final — Ciencia de Datos: Base de Datos Cafetería
### Inspección, Perfilado y Preprocesamiento
**Asignatura:** Introducción a la Ciencia de Datos  
**Programa:** Ingeniería Electrónica — ITM  
**Variable objetivo:** `Propina`


## 1. Importar Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11


## 2. Carga del Dataset

In [ ]:
df = pd.read_excel("Base_de_datos_Cafeteria.xlsx")

print(f"✅ Dataset cargado correctamente")
print(f"   Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head(3)


## 3. Inspección Técnica y Perfilado Estadístico Inicial
Evaluamos dimensiones, tipos de datos, valores nulos, duplicados y estadísticas descriptivas.


In [ ]:
print("=" * 55)
print("  INSPECCIÓN GENERAL DEL DATASET")
print("=" * 55)
print(f"  Filas       : {df.shape[0]:,}")
print(f"  Columnas    : {df.shape[1]}")
print(f"  Duplicados  : {df.duplicated().sum()}")
print(f"  Valores nulos totales: {df.isnull().sum().sum()}")
print("=" * 55)


In [ ]:
# Tipos de datos y valores nulos por columna
info = pd.DataFrame({
    'Tipo': df.dtypes,
    'Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df) * 100).round(2),
    'Únicos': df.nunique()
})
print(info.to_string())


In [ ]:
# Estadísticas descriptivas de variables numéricas
df.describe().round(4)


### 3.1 Distribución de la Variable Objetivo: Propina

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Propina'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de Propina (antes de limpieza)')
axes[0].set_xlabel('Propina (fracción del precio)')
axes[0].set_ylabel('Frecuencia')

# Boxplot
axes[1].boxplot(df['Propina'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Boxplot de Propina')
axes[1].set_ylabel('Propina')

plt.tight_layout()
plt.savefig('propina_dist_inicial.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Rango Propina: [{df['Propina'].min():.4f}, {df['Propina'].max():.4f}]")
print(f"Media: {df['Propina'].mean():.4f} | Mediana: {df['Propina'].median():.4f}")


### 3.2 Detección de Outliers en Variable `Cantidad`

In [ ]:
Q1 = df['Cantidad'].quantile(0.25)
Q3 = df['Cantidad'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['Cantidad'] < lower) | (df['Cantidad'] > upper)]
print(f"Rango IQR: [{lower:.2f}, {upper:.2f}]")
print(f"Outliers detectados: {len(outliers):,} ({len(outliers)/len(df)*100:.2f}%)")

fig, ax = plt.subplots(figsize=(10, 4))
ax.boxplot(df['Cantidad'], vert=False, patch_artist=True,
           boxprops=dict(facecolor='salmon', alpha=0.7))
ax.set_title('Outliers en Cantidad (antes de limpieza)')
ax.set_xlabel('Cantidad')
plt.tight_layout()
plt.savefig('outliers_cantidad_inicial.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.3 Variable `Hora de Cobro` — Análisis de formato

In [ ]:
# La hora está en formato decimal (fracción del día): 0.55 = 13:12, 1.0 = 24:00
print("Hora de Cobro — primeras filas (formato decimal):")
print(df['Hora de Cobro'].head(10).values)

# Convertir a hora real (HH:MM)
df['Hora_decimal'] = df['Hora de Cobro']
df['Hora_real'] = pd.to_datetime(df['Hora de Cobro'] * 24, unit='h', origin='2000-01-01').dt.strftime('%H:%M')

print("\nEjemplo de conversión:")
print(df[['Hora de Cobro', 'Hora_real']].head(8).to_string(index=False))


### 3.4 Matriz de Correlación Inicial

In [ ]:
num_cols = ['Precio', 'Cantidad', 'Propina', 'Hora de Cobro', 'Mesa']
corr = df[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True)
plt.title('Mapa de Calor — Correlaciones Iniciales')
plt.tight_layout()
plt.savefig('heatmap_inicial.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nCorrelaciones con Propina:")
print(corr['Propina'].sort_values(ascending=False).to_string())


## 4. Preprocesamiento y Limpieza de Datos
Se identificaron los siguientes problemas a corregir:
| # | Problema | Estrategia | Justificación |
|---|----------|-----------|---------------|
| 1 | 1 fila duplicada | Eliminación | Introduce sesgo en el análisis |
| 2 | Typo en `Tipode`: "Recuerrente" | Corrección de texto | Error ortográfico evidente |
| 3 | `Hora de Cobro` en formato decimal | Conversión a horas reales | Mejor interpretabilidad |
| 4 | `Propina` como fracción | Conversión a pesos COP | Coherencia con `Precio` |
| 5 | Outliers en `Cantidad` (IQR) | Filtrado por IQR | Valores extremos distorsionan modelos |


In [ ]:
df_clean = df.copy()

# ── Paso 1: Eliminar duplicados ──────────────────────────────────────────────
n_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"✅ Duplicados eliminados: {n_before - len(df_clean)}")

# ── Paso 2: Corregir typo en columna 'Tipode' ────────────────────────────────
df_clean['Tipode'] = df_clean['Tipode'].replace('Recuerrente', 'Recurrente')
print(f"✅ Typo corregido en 'Tipode': {df_clean['Tipode'].value_counts().to_dict()}")

# ── Paso 3: Convertir Hora de Cobro decimal → hora entera (0-23) ─────────────
df_clean['Hora de Cobro'] = (df_clean['Hora de Cobro'] * 24).round(0).astype(int).clip(0, 23)
print(f"✅ Hora de Cobro convertida. Rango: [{df_clean['Hora de Cobro'].min()}, {df_clean['Hora de Cobro'].max()}]")

# ── Paso 4: Convertir Propina de fracción a pesos COP ────────────────────────
df_clean['Propina'] = (df_clean['Propina'] * df_clean['Precio']).round(0).astype(int)
print(f"✅ Propina convertida a COP. Rango: [{df_clean['Propina'].min():,}, {df_clean['Propina'].max():,}]")

# ── Paso 5: Eliminar outliers en Cantidad usando IQR ─────────────────────────
Q1 = df_clean['Cantidad'].quantile(0.25)
Q3 = df_clean['Cantidad'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
n_before = len(df_clean)
df_clean = df_clean[(df_clean['Cantidad'] >= lower) & (df_clean['Cantidad'] <= upper)]
print(f"✅ Outliers de Cantidad eliminados: {n_before - len(df_clean):,} filas")

# ── Columna Hora_decimal y Hora_real (auxiliares — eliminar) ─────────────────
if 'Hora_decimal' in df_clean.columns:
    df_clean = df_clean.drop(columns=['Hora_decimal', 'Hora_real'])

print(f"\n📊 Dataset limpio: {df_clean.shape[0]:,} filas × {df_clean.shape[1]} columnas")


## 5. Perfilado Post-Limpieza y Comparación

In [ ]:
print("=" * 50)
print("  COMPARACIÓN ANTES vs DESPUÉS")
print("=" * 50)
print(f"  Filas antes  : {df.shape[0]:,}")
print(f"  Filas después: {df_clean.shape[0]:,}  (−{df.shape[0]-df_clean.shape[0]:,})")
print(f"  Duplicados   : {df.duplicated().sum()} → 0")
print(f"  Typos Tipode : 'Recuerrente' → 'Recurrente'")
print(f"  Propina      : fracción → COP")
print(f"  Hora Cobro   : decimal  → entero (0-23)")
print("=" * 50)


In [ ]:
# Distribución Propina — Antes vs Después
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Propina'], bins=30, color='tomato', edgecolor='white', alpha=0.8)
axes[0].set_title('Propina ANTES (fracción del precio)')
axes[0].set_xlabel('Propina')

axes[1].hist(df_clean['Propina'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Propina DESPUÉS (pesos COP)')
axes[1].set_xlabel('Propina (COP)')

plt.suptitle('Variable Objetivo: Propina — Antes vs Después de Limpieza', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('propina_comparacion.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Heatmap post-limpieza
num_cols_clean = ['Precio', 'Cantidad', 'Propina', 'Hora de Cobro', 'Mesa']
corr_clean = df_clean[num_cols_clean].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_clean, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True)
plt.title('Mapa de Calor — Correlaciones Post-Limpieza')
plt.tight_layout()
plt.savefig('heatmap_limpio.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCorrelaciones con Propina (post-limpieza):")
print(corr_clean['Propina'].sort_values(ascending=False).to_string())


In [ ]:
# Distribución de variables categóricas clave
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df_clean['Vendedor'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Ventas por Vendedor')
axes[0].set_ylabel('Número de transacciones')
axes[0].tick_params(axis='x', rotation=30)

df_clean['Tipo de pago'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                              colors=['steelblue', 'salmon'])
axes[1].set_title('Tipo de Pago')
axes[1].set_ylabel('')

df_clean['Zona'].value_counts().plot(kind='barh', ax=axes[2], color='mediumseagreen', edgecolor='white')
axes[2].set_title('Transacciones por Zona')
axes[2].set_xlabel('Cantidad')

plt.suptitle('Variables Categóricas Principales (Dataset Limpio)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('categoricas_limpias.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Propina promedio por vendedor
propina_vendedor = df_clean.groupby('Vendedor')['Propina'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
propina_vendedor.plot(kind='bar', color='goldenrod', edgecolor='white')
plt.title('Propina Promedio por Vendedor (COP)')
plt.ylabel('Propina promedio (COP)')
plt.xlabel('Vendedor')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('propina_por_vendedor.png', dpi=150, bbox_inches='tight')
plt.show()
print(propina_vendedor.to_string())


## 6. Exportar Dataset Limpio

In [ ]:
df_clean.to_csv("cafeteria_limpio.csv", index=False, encoding='utf-8-sig')
print(f"✅ Dataset exportado: 'cafeteria_limpio.csv'")
print(f"   Filas: {df_clean.shape[0]:,} | Columnas: {df_clean.shape[1]}")
print(f"\n--- RESUMEN FINAL ---")
df_clean.describe().round(2)
